# Cornell Tech CS5785 · Human or AI?
Predict the probability of AI authorship (label 1). Kaggle ranks the 3,000 test predictions by AUROC. The private split accounts for 70% of the test rows. See the [competition overview](https://www.kaggle.com/competitions/cornell-tech-cs-5785-combined-2026-ai-vs-human-text/overview) and [rules](https://www.kaggle.com/competitions/cornell-tech-cs-5785-combined-2026-ai-vs-human-text/rules).

**In Colab**, upload this notebook, select a GPU runtime for the optional encoder, and upload the Kaggle ZIP through the Files sidebar. Set `DATA_SOURCE` to its path. The CPU pipeline can run without a GPU. The encoder runs inside the notebook runtime using downloaded weights. Confirm that the course permits pretrained weights and cloud processing before enabling it.

The notebook uses the supplied labeled and unlabeled CSVs, with test text reserved for inference. It exports predictions locally and records the validation results. Keep the competition data and outputs within the team. Review the results and document your modeling decisions and AI assistance in the course report.

In [ ]:
%pip install -q scikit-learn==1.9.1 catboost==1.2.10

In [ ]:
from pathlib import Path
import math
import re
import zlib
from collections import Counter
from zipfile import ZipFile

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

DATA_SOURCE = Path('/content/cornell-tech-cs-5785-combined-2026-ai-vs-human-text.zip')
OUTPUT_DIR = Path('results')
RUN_ENCODER = False
SEED = 5785
OUTPUT_DIR.mkdir(exist_ok=True)

def read_table(name):
    if DATA_SOURCE.is_dir():
        return pd.read_csv(DATA_SOURCE / name, keep_default_na=False)
    with ZipFile(DATA_SOURCE) as archive, archive.open(name) as handle:
        return pd.read_csv(handle, keep_default_na=False)

train, unlabeled, test, sample, starter = [
    read_table(name) for name in (
        'train.csv', 'unlabeled.csv', 'test.csv', 'sample_submission.csv',
        'train_starter_2000.csv',
    )
]
assert (len(train), len(unlabeled), len(test), len(sample), len(starter)) == (10000, 3000, 3000, 3000, 2000)
assert train.label.value_counts().to_dict() == {0: 5000, 1: 5000}
assert all(frame.id.is_unique for frame in (train, unlabeled, test, sample, starter))
assert set(sample.id) == set(test.id)
assert set(starter.id) <= set(train.id)
assert all(frame.text.str.strip().ne('').all() for frame in (train, unlabeled, test))

data = pd.concat([train, unlabeled, test], ignore_index=True)
texts = data.text.to_numpy()
y = train.label.to_numpy()
labeled_idx = np.arange(len(train))
unlabeled_idx = np.arange(len(train), len(train) + len(unlabeled))
test_idx = np.arange(len(train) + len(unlabeled), len(data))
normalized = data.text.str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
# Avoid using an unlabeled exact duplicate of a labeled text in self-training.
unlabeled_idx = unlabeled_idx[~normalized.iloc[unlabeled_idx].isin(normalized.iloc[labeled_idx]).to_numpy()]
print('Usable unlabeled rows:', len(unlabeled_idx))

## Validation design

Reserve an audit split before model selection. On the remaining examples, group passages by coarse topic using content-word TF-IDF, SVD and clustering. Use whole clusters for validation to test transfer across topics. These clusters approximate domains because Kaggle supplies no domain or generator labels. Exact normalized duplicates are kept together in the audit split. The selection split controls model choice and the audit split measures the selected recipe once. Neither estimates the score on undisclosed generators directly.

In [ ]:
duplicate_groups = pd.factorize(normalized.iloc[labeled_idx])[0]
if len(np.unique(duplicate_groups)) == len(labeled_idx):
    dev, audit = train_test_split(labeled_idx, test_size=.2, stratify=y, random_state=SEED)
else:
    dev, audit = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(
        labeled_idx, y, duplicate_groups))
topic_words = TfidfVectorizer(stop_words='english', min_df=4, max_features=24000)
topic_vectors = topic_words.fit_transform(texts[dev])
embedding = TruncatedSVD(40, random_state=SEED).fit_transform(topic_vectors)
groups = MiniBatchKMeans(30, n_init=5, random_state=SEED).fit_predict(embedding)
# Exact duplicates have identical embeddings and cluster assignments.
topic_splits = list(StratifiedGroupKFold(4, shuffle=True, random_state=SEED).split(dev, y[dev], groups))
topic_splits = [(dev[tr], dev[va]) for tr, va in topic_splits]
print('Topic fold sizes:', [len(va) for _, va in topic_splits])

## Features and models

The lexical model combines word and character TF-IDF. The style model uses CatBoost over punctuation, casing, function words, sentence rhythm, lexical repetition, and compression. The blend adds complementary evidence from the two models. The self-training variant learns from high-confidence agreements on the supplied unlabeled passages and gives them lower weight than labeled rows.

In [ ]:
WORD = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?")
FUNCTION_WORDS = sorted(ENGLISH_STOP_WORDS)
PUNCTUATION = ".,!?;:-—–'’“”()[]/#*%&" + chr(34) + "\n\t"

def distribution(values):
    if not values:
        return [0.] * 5
    return [np.mean(values), np.std(values), *np.quantile(values, [.1, .5, .9])]

def style_features(text):
    words = WORD.findall(text)
    lower = [word.lower() for word in words]
    counts = Counter(lower)
    chars = Counter(text)
    n_chars, n_words = max(1, len(text)), max(1, len(words))
    sentences = [part for part in re.split(r'(?<=[.!?])\s+|\n+', text) if part.strip()]
    bigrams = Counter(zip(lower, lower[1:]))
    data_bytes = text.encode('utf-8')
    case = [sum(word.isupper() for word in words), sum(word.istitle() for word in words)]
    return [
        math.log1p(len(text)), math.log1p(len(words)), len(sentences),
        len(counts) / n_words, len(counts) / math.sqrt(n_words),
        sum(v == 1 for v in counts.values()) / n_words,
        max(counts.values(), default=0) / n_words,
        len(bigrams) / n_words, sum(v > 1 for v in bigrams.values()) / n_words,
        len(zlib.compress(data_bytes)) / max(1, len(data_bytes)),
        len(zlib.compress(' '.join(lower).encode())) / n_words,
        sum(c.isdigit() for c in text) / n_chars, sum(ord(c) > 127 for c in text) / n_chars,
        sum(c.isupper() for c in text) / n_chars, *[n / n_words for n in case],
        *distribution([len(w) for w in words]),
        *distribution([len(WORD.findall(s)) for s in sentences]),
        *distribution([len(s) for s in text.splitlines() if s.strip()]),
        *[chars[c] / n_chars for c in PUNCTUATION],
        *[counts[w] / n_words for w in FUNCTION_WORDS],
    ]

style = np.asarray([style_features(t) for t in texts], dtype=np.float32)
print('Style matrix:', style.shape)

In [ ]:
def model_scores(fit_idx, pred_idx, targets=None, weights=None):
    from scipy.sparse import hstack

    targets = y[fit_idx] if targets is None else targets
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=80000,
                           sublinear_tf=True, strip_accents='unicode', dtype=np.float32)
    char = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=3, max_features=100000,
                           sublinear_tf=True, dtype=np.float32)
    x_word = word.fit_transform(texts[fit_idx])
    x_char = char.fit_transform(texts[fit_idx])
    lexical = LogisticRegression(C=8, solver='liblinear', max_iter=500, random_state=SEED)
    lexical.fit(hstack((x_word, x_char), format='csr'), targets, sample_weight=weights)
    x_pred = hstack((word.transform(texts[pred_idx]), char.transform(texts[pred_idx])), format='csr')
    stylistic = CatBoostClassifier(iterations=850, depth=6, learning_rate=.06,
                                   verbose=False, thread_count=5, random_seed=SEED,
                                   allow_writing_files=False)
    stylistic.fit(style[fit_idx], targets, sample_weight=weights)
    return {
        'lexical': lexical.decision_function(x_pred),
        'style': stylistic.predict(style[pred_idx], prediction_type='RawFormulaVal'),
    }

def combine(score):
    return .35 * score['lexical'] + .65 * score['style']

def classical(fit_idx, pred_idx, try_pseudo=True):
    extra = unlabeled_idx if try_pseudo else np.array([], dtype=int)
    scores = model_scores(fit_idx, np.r_[pred_idx, extra])
    result = {name: values[:len(pred_idx)] for name, values in scores.items()}
    result['blend'] = combine(result)
    if not try_pseudo:
        return result, 0
    candidates = {name: values[len(pred_idx):] for name, values in scores.items()}
    a, b = candidates['lexical'], candidates['style']
    strength = np.minimum(np.abs(a), np.abs(b))
    picks = [
        idx[np.argsort(strength[idx])[-450:]]
        for direction in (-1, 1)
        for idx in [np.flatnonzero((direction * a > 1.35) & (direction * b > 1.35))]
    ]
    chosen = np.concatenate(picks)
    if min(map(len, picks)) < 25:
        result['pseudo_blend'] = result['blend']
        return result, 0
    pseudo_targets = (a[chosen] > 0).astype(int)
    refit = model_scores(
        np.r_[fit_idx, extra[chosen]], pred_idx,
        np.r_[y[fit_idx], pseudo_targets],
        np.r_[np.ones(len(fit_idx)), np.full(len(chosen), .35)],
    )
    result['pseudo_blend'] = .5 * result['blend'] + .5 * combine(refit)
    return result, len(chosen)

In [ ]:
# Run on topic-held-out folds. Set to 4 to run every topic fold.
TOPIC_FOLDS = 3
records = []
for fold, (fit_idx, val_idx) in enumerate(topic_splits[:TOPIC_FOLDS]):
    scores, pseudo_count = classical(fit_idx, val_idx)
    for name, values in scores.items():
        records.append({'fold': fold, 'model': name, 'auroc': roc_auc_score(y[val_idx], values),
                        'pseudo_rows': pseudo_count if name == 'pseudo_blend' else 0})
    print(pd.DataFrame(records).query('fold == @fold').to_string(index=False), flush=True)

topic_results = pd.DataFrame(records)
topic_results.to_csv(OUTPUT_DIR / 'topic_validation.csv', index=False)
ranking = topic_results.groupby('model').auroc.agg(['mean', 'min']).sort_values('mean', ascending=False)
# Prefer a supervised recipe if pseudo-labeling yields a negligible mean gain.
best_cpu = ranking.drop(index='pseudo_blend')['mean'].idxmax()
if ranking.loc['pseudo_blend', 'mean'] > ranking.loc[best_cpu, 'mean'] + .001:
    best_cpu = 'pseudo_blend'
print(ranking, '\nSelected CPU recipe:', best_cpu)

In [ ]:
# Report experiment: compare nested training sizes on the same topic split.
fit_idx, val_idx = topic_splits[0]
small = fit_idx[np.isin(train.id.iloc[fit_idx], starter.id)]
remaining = np.setdiff1d(fit_idx, small)
additions, _ = train_test_split(remaining, train_size=4000-len(small),
                                stratify=y[remaining], random_state=SEED)
middle = np.r_[small, additions]
size_results = []
for subset in (small, middle):
    score, _ = classical(subset, val_idx, try_pseudo=False)
    size_results.append({'rows': len(subset), 'blend_auroc': roc_auc_score(y[val_idx], score['blend'])})
full_auc = topic_results.query("fold == 0 and model == 'blend'").auroc.iloc[0]
size_results.append({'rows': len(fit_idx), 'blend_auroc': full_auc})
pd.DataFrame(size_results).to_csv(OUTPUT_DIR / 'learning_curve.csv', index=False)
print(pd.DataFrame(size_results))

## Formatting sensitivity and backup model

Compare the original style classifier against a blend with a second classifier trained on whitespace-normalized text. Collapse whitespace at validation time to simulate a preprocessing change. The normalized component keeps the same representation under that change. The backup uses a fixed 40% weight on the normalized component. The original CPU selection above uses the unmodified topic folds.

In [ ]:
flat_style = np.asarray([
    style_features(re.sub(r'\s+', ' ', t).strip()) for t in texts
], dtype=np.float32)
FORMAT_WEIGHT = .4

def fit_style(matrix, fit_idx):
    model = CatBoostClassifier(iterations=850, depth=6, learning_rate=.06,
                               verbose=False, thread_count=5, random_seed=SEED,
                               allow_writing_files=False)
    return model.fit(matrix[fit_idx], y[fit_idx])

def raw_style(model, matrix, pred_idx):
    return model.predict(matrix[pred_idx], prediction_type='RawFormulaVal')

def formatting_scores(fit_idx, pred_idx):
    original = fit_style(style, fit_idx)
    normalized = fit_style(flat_style, fit_idx)
    clean = raw_style(original, style, pred_idx)
    shifted = raw_style(original, flat_style, pred_idx)
    stable = raw_style(normalized, flat_style, pred_idx)
    return {
        'original': clean,
        'original_shifted': shifted,
        'backup': (1-FORMAT_WEIGHT) * clean + FORMAT_WEIGHT * stable,
        'backup_shifted': (1-FORMAT_WEIGHT) * shifted + FORMAT_WEIGHT * stable,
    }

format_results = []
for fold, (fit_idx, val_idx) in enumerate(topic_splits[:TOPIC_FOLDS]):
    for name, scores in formatting_scores(fit_idx, val_idx).items():
        format_results.append({'fold': fold, 'condition': name,
                               'auroc': roc_auc_score(y[val_idx], scores)})
format_results = pd.DataFrame(format_results)
print(format_results.groupby('condition').auroc.agg(['mean', 'min']))
format_results.to_csv(OUTPUT_DIR / 'format_stress.csv', index=False)

## Optional Colab GPU experiment

Fine-tune the [MIT-licensed DeBERTa-v3-base encoder](https://huggingface.co/microsoft/deberta-v3-base) on the course labels. Keep tokens from the beginning and end for long passages. Compare it with the selected CPU recipe on two topic folds before opening the audit results. If enabled, the encoder is evaluated locally in the Colab runtime. No inference endpoint is called.

In [ ]:
if RUN_ENCODER:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'transformers==4.51.3', 'accelerate==1.7.0', 'sentencepiece'])

In [ ]:
if RUN_ENCODER:
    import gc
    import tempfile
    import torch
    from transformers import (
        AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
        Trainer, TrainingArguments, set_seed,
    )

    if not torch.cuda.is_available():
        raise RuntimeError('Select a Colab GPU runtime before enabling RUN_ENCODER.')
    MODEL = 'microsoft/deberta-v3-base'
    tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=False)
    token_limit = 384
    budget = token_limit - tokenizer.num_special_tokens_to_add(pair=False)

    class Passages(torch.utils.data.Dataset):
        def __init__(self, indices, labeled=False):
            self.items = []
            for index in indices:
                ids = tokenizer.encode(texts[index], add_special_tokens=False, verbose=False)
                if len(ids) > budget:
                    tail = budget // 3
                    ids = ids[:budget-tail] + ids[-tail:]
                item = {'input_ids': tokenizer.build_inputs_with_special_tokens(ids)}
                if labeled:
                    item['labels'] = int(y[index])
                self.items.append(item)

        def __len__(self):
            return len(self.items)

        def __getitem__(self, index):
            return self.items[index]

    def encoder_scores(fit_idx, pred_idx, seed=SEED):
        set_seed(seed)
        bf16 = torch.cuda.is_bf16_supported()
        model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
        with tempfile.TemporaryDirectory() as tmp:
            arguments = TrainingArguments(
                output_dir=tmp, learning_rate=2e-5, weight_decay=.01, num_train_epochs=2,
                warmup_ratio=.08, per_device_train_batch_size=4,
                gradient_accumulation_steps=8, per_device_eval_batch_size=8,
                gradient_checkpointing=True, bf16=bf16, fp16=False,
                save_strategy='no', report_to='none', logging_steps=50, seed=seed,
            )
            trainer = Trainer(
                model=model, args=arguments, train_dataset=Passages(fit_idx, labeled=True),
                data_collator=DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8),
            )
            trainer.train()
            logits = trainer.predict(Passages(pred_idx)).predictions.astype(float)
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()
        return logits[:, 1] - logits[:, 0]

encoder_weight = 0.
if RUN_ENCODER:
    candidates = []
    for fold, (fit_idx, val_idx) in enumerate(topic_splits[:2]):
        cpu, _ = classical(fit_idx, val_idx, try_pseudo=best_cpu == 'pseudo_blend')
        gpu = encoder_scores(fit_idx, val_idx)
        for weight in (0., .35, .65, 1.):
            mixed = (1-weight) * cpu[best_cpu] + weight * gpu
            candidates.append({'fold': fold, 'encoder_weight': weight,
                               'auroc': roc_auc_score(y[val_idx], mixed)})
    encoder_results = pd.DataFrame(candidates)
    encoder_results.to_csv(OUTPUT_DIR / 'encoder_validation.csv', index=False)
    encoder_ranking = encoder_results.groupby('encoder_weight').auroc.mean()
    encoder_weight = float(encoder_ranking.idxmax())
    if encoder_ranking[encoder_weight] <= encoder_ranking[0.] + .001:
        encoder_weight = 0.
    print(encoder_ranking, 'Selected encoder weight:', encoder_weight)

## Audit and final prediction

The audit is held out from recipe selection. Inspect errors by passage length and review representative mistakes using the original labeled CSV. Calibration is fitted on audit predictions only after measuring AUROC. All final model components are refit on the full labeled training set.

In [ ]:
def selected_scores(fit_idx, pred_idx):
    cpu, count = classical(fit_idx, pred_idx, try_pseudo=best_cpu == 'pseudo_blend')
    values = cpu[best_cpu]
    if encoder_weight:
        gpu = encoder_scores(fit_idx, pred_idx)
        values = (1-encoder_weight) * values + encoder_weight * gpu
    return values, count

audit_logits, _ = selected_scores(dev, audit)
print('Audit AUROC:', roc_auc_score(y[audit], audit_logits))
calibrator = LogisticRegression(C=100).fit(audit_logits.reshape(-1, 1), y[audit])
assert calibrator.coef_[0, 0] > 0, 'Audit scores are inverted.'
audit_rows = pd.DataFrame({
    'id': train.id.iloc[audit].to_numpy(), 'label': y[audit], 'score': audit_logits,
    'words': [len(WORD.findall(t)) for t in texts[audit]],
})
audit_rows['length_bin'] = pd.qcut(audit_rows.words, 4, duplicates='drop').astype(str)
audit_rows['error_confidence'] = np.where(audit_rows.label == 0, audit_logits, -audit_logits)
audit_rows.sort_values('error_confidence', ascending=False).to_csv(OUTPUT_DIR / 'audit_predictions.csv', index=False)
length_rows = [
    {'length_bin': band, 'rows': len(frame), 'auroc': roc_auc_score(frame.label, frame.score)}
    for band, frame in audit_rows.groupby('length_bin') if frame.label.nunique() == 2
]
pd.DataFrame(length_rows).to_csv(OUTPUT_DIR / 'audit_length.csv', index=False)
print(pd.DataFrame(length_rows))

In [ ]:
final_logits, final_pseudo_rows = selected_scores(labeled_idx, test_idx)
prediction = calibrator.predict_proba(final_logits.reshape(-1, 1))[:, 1]
by_id = pd.Series(prediction, index=test.id)
submission = pd.DataFrame({'id': sample.id, 'label': sample.id.map(by_id)})
assert len(submission) == 3000 and submission.id.is_unique
assert submission.id.equals(sample.id) and list(submission) == ['id', 'label']
assert np.isfinite(submission.label).all() and submission.label.between(0, 1).all()
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print('Final recipe:', best_cpu, 'Encoder weight:', encoder_weight, 'Pseudo rows:', final_pseudo_rows)
print('Local output:', OUTPUT_DIR / 'submission.csv')

In [ ]:
# Backup CSV for comparing the effect of formatting robustness.
backup_audit = formatting_scores(dev, audit)['backup']
print('Backup audit AUROC:', roc_auc_score(y[audit], backup_audit))
backup_calibration = LogisticRegression(C=100).fit(backup_audit.reshape(-1, 1), y[audit])
assert backup_calibration.coef_[0, 0] > 0
backup_test = formatting_scores(labeled_idx, test_idx)['backup']
backup_probability = backup_calibration.predict_proba(backup_test.reshape(-1, 1))[:, 1]
backup = pd.DataFrame({'id': sample.id, 'label': sample.id.map(pd.Series(backup_probability, index=test.id))})
assert len(backup) == 3000 and backup.id.equals(sample.id)
assert np.isfinite(backup.label).all() and backup.label.between(0, 1).all()
backup.to_csv(OUTPUT_DIR / 'submission_format_backup.csv', index=False)